## Citation Network Node Classification — Exploration

In [ ]:
import sys
sys.path.append('../src')

import torch
import matplotlib.pyplot as plt

from data_loader import load_dataset
import graph_analysis as ga


## 1. Load the dataset

In [ ]:
dataset, data = load_dataset('Cora', root='../data')
print('Features:', dataset.num_features, ', classes:', dataset.num_classes)
data

## 2. Structural analysis


In [ ]:
ga.print_report(data, 'Cora')

In [ ]:
dist = ga.class_distribution(data)
plt.bar([str(k) for k in dist.keys()], list(dist.values()))
plt.xlabel('class'); plt.ylabel('number of nodes'); plt.title('Cora class distribution')
plt.show()

## 3. Compare baselines and GNNs

In [ ]:
from logreg_baseline import run_logistic_regression
from mlp_baseline import MLP
from gcn import GCN
from graphsage import GraphSAGE
from gat import GAT
from train import train_model

torch.manual_seed(42)
inc, outc = dataset.num_features, dataset.num_classes

results = {'LogReg': run_logistic_regression(data)['test']}
models = {
    'MLP': MLP(inc, 64, outc),
    'GCN': GCN(inc, 64, outc),
    'GraphSAGE': GraphSAGE(inc, 64, outc),
    'GAT': GAT(inc, 64, outc),
}
for name, m in models.items():
    results[name] = train_model(m, data, epochs=200)['best_test_acc']

for name, acc in results.items():
    print(f'{name:<12} {acc:.4f}')

In [ ]:
plt.bar(list(results.keys()), list(results.values()))
plt.ylabel('test accuracy'); plt.title('Model comparison on Cora'); plt.ylim(0, 1)
plt.show()

## 4. t-SNE

In [ ]:
from sklearn.manifold import TSNE

gcn = GCN(inc, 64, outc)
train_model(gcn, data, epochs=200)
gcn.eval()
with torch.no_grad():
    emb = gcn.embed(data.x, data.edge_index).numpy()

emb2d = TSNE(n_components=2, init='pca', random_state=42).fit_transform(emb)
sc = plt.scatter(emb2d[:,0], emb2d[:,1], c=data.y.numpy(), cmap='tab10', s=10)
plt.legend(*sc.legend_elements(), title='Class', fontsize=8)
plt.xticks([]); plt.yticks([]); plt.title('t-SNE of GCN embeddings (Cora)')
plt.show()